# 02 · Connectivity — which connections are necessary?

*Task 02 of the rebuilt mRNN analysis. Width is fixed by task 01; this asks what has to be
wired to what.*

**The baseline is the unconstrained model**: every region connected to every other, with
dense full-rank inter-regional blocks and no bottleneck. Each variant then removes
something and is **refitted from scratch**, so the question is always "can the data still
be reproduced without this?" rather than "how much does breaking a fitted model hurt?".

Those are different questions and the legacy analysis conflated them. Deleting a block
from an already-fitted network drove $R^2$ from 0.998 to about −2 for *every* block, which
says only that the readouts were tuned to the intact dynamics. Refitting asks the question
that has an interpretable answer.

| Arm | What it removes | Asks |
|---|---|---|
| baseline | nothing | how well can this be fitted at all |
| global structure | all cross-region blocks, or the dense within-region blocks | do regions need each other, and do they need internal recurrence |
| region isolation | every connection into and out of one region | can the other three be reproduced without it |
| pathway removal | one directed pathway | is any single route load-bearing |

Every arm is scored the same way: fit against the measured noise ceiling, the parameter
count that bought it, agreement across seeds, and a visual check of the traces.

| Section | |
|---|---|
| 1 | What task 01 settled, and the variants |
| 2 | Run state and submission (off by default) |
| 3 | Loss trajectories and convergence |
| 4 | Fit against the ceiling, and parameter cost |
| 5 | The visual check across the sweep |
| 6 | Seed agreement |
| 7 | Reading the result |

In [1]:
from __future__ import annotations

from pathlib import Path
import sys

import numpy as np
import pandas as pd
import yaml
from IPython.display import Image, Markdown, display

repo_root = Path.cwd()
if not (repo_root / "src").exists():
    repo_root = next(parent for parent in Path.cwd().parents if (parent / "src").exists())
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

from dal_monte_2022_analysis.config.load import load_config
from dal_monte_2022_analysis.ephys.analysis import fixation_mrnn_protocol as protocol
from dal_monte_2022_analysis.ephys.analysis import fixation_mrnn_sweep as sweep
from dal_monte_2022_analysis.ephys.analysis import fixation_mrnn_synthesis as syn
from dal_monte_2022_analysis.ephys.analysis import fixation_mrnn_target_loss as tl
from dal_monte_2022_analysis.ephys.analysis import fixation_psth_noise_ceiling as ceiling_mod
from dal_monte_2022_analysis.ephys.plotting import fixation_mrnn_sweep as viz
from dal_monte_2022_analysis.ephys.plotting.thesis_common import (
    ThesisFigureSettings,
    apply_thesis_plot_style,
    figure_to_png_bytes,
    save_thesis_figure,
)

DATASET_CFG_PATH = repo_root / "configs" / "dataset.yaml"
MRNN_CFG_PATH = repo_root / "configs" / "ephys_fixation_mrnn.yaml"
apply_thesis_plot_style(load_config(repo_root / "configs" / "plotting.yaml"))

PROTOCOL_ROOT = protocol.resolve_chapter_root(DATASET_CFG_PATH, task="00_training_protocol")
CAPACITY_ROOT = sweep.resolve_task_root("01_capacity", DATASET_CFG_PATH)
TASK_ROOT = sweep.resolve_task_root("02_connectivity", DATASET_CFG_PATH)
CEILING_DIR = ceiling_mod.resolve_output_dir(DATASET_CFG_PATH)
FIGURE_DIR = syn.resolve_output_dir(DATASET_CFG_PATH, scope="02_connectivity")
FIGURES = ThesisFigureSettings(output_dir=FIGURE_DIR)

SELECTED_PROTOCOL = sweep.load_selected_protocol(PROTOCOL_ROOT / "selected_protocol.yaml")
REGIONS = tuple(SELECTED_PROTOCOL["architecture"].get("region_order", ("ofc", "bla", "dmpfc", "accg")))

#: Width comes from task 01. Until that sweep finishes the notebook falls back to 50 and
#: says so, so it can be read and reviewed before the dependency lands.
CAPACITY_PATH = CAPACITY_ROOT / "selected_capacity.yaml"
if CAPACITY_PATH.exists():
    SELECTED_CAPACITY = yaml.safe_load(CAPACITY_PATH.read_text())
    HIDDEN_UNITS = int(SELECTED_CAPACITY["hidden_units"])
    CAPACITY_SOURCE = f"task 01 (`{SELECTED_CAPACITY['selected_label']}`)"
else:
    SELECTED_CAPACITY = None
    HIDDEN_UNITS = 50
    CAPACITY_SOURCE = "**provisional fallback** — task 01 has not finished"

#: Set True to queue this sweep before task 01 has chosen a width. Cluster time is the
#: scarce resource and these arrays take hours, so waiting for a clean dependency can cost
#: more than the risk: if task 01 selects a different width, the runs fitted here are at
#: the wrong one and Section 1 will say so. The width every cell was actually trained at is
#: recorded in its own run_config.yaml either way, so a mismatch is always detectable.
ALLOW_PROVISIONAL_WIDTH = False

SWEEP_SEEDS = 3
GALLERY_REGION = "ofc"


def show(figure, stem: str) -> None:
    save_thesis_figure(figure, FIGURES, stem)
    display(Image(data=figure_to_png_bytes(figure, dpi=190)))


print("regions      :", REGIONS)
print("hidden units :", HIDDEN_UNITS)
print("task root    :", TASK_ROOT)

regions      : ('ofc', 'bla', 'dmpfc', 'accg')
hidden units : 40
task root    : /gpfs/milgram/pi/chang/pg496/repositories/local_data/dal_monte_2022/analysis_outputs/ephys/modeling/fixation_mrnn/chapter/02_connectivity


## 1. The variants

All four arms share the width from task 01, the recipe from task 00, and dense
inter-regional blocks. Only the connectivity differs.

The **region isolation** arm removes every connection into *and* out of one region. That
region keeps its own within-region block, its condition input, its trained initial state
and its readout — it is still fitted, just autonomously — while the other three keep
talking to each other.

> **The loss is a sum over all four regions' readouts, so a single pooled score cannot
> read this arm.** Cutting a region off changes two things at once: the isolated region
> now has to reproduce its own trajectories without the others' input, *and* the remaining
> three have to reproduce theirs without its input. Those are different questions — a
> region can be perfectly reproducible alone while being indispensable to everyone else,
> or the reverse. Section 4a therefore reports the two costs **separately**, and only the
> second speaks to whether the region is necessary to the rest of the network.

The **pathway removal** arm takes out one directed block at a time. Twelve pathways is a
lot of fitting, so the default is the four into and out of BLA — the region the legacy
ensemble singled out as changing its drive share during interactive-face fixations. Widen
`PATHWAYS_TO_TEST` if the result warrants it.

In [2]:
PATHWAYS_TO_TEST = [(source, "bla") for source in REGIONS if source != "bla"] + \
                   [("bla", target) for target in REGIONS if target != "bla"]

base_overrides = {"hidden_units": HIDDEN_UNITS, "recurrent_bottleneck_dim": None}

variants = [
    sweep.ModelVariant(label="full", arm="baseline",
                       overrides={**base_overrides, "recurrent_connectivity": "full"}),
    sweep.ModelVariant(label="within_region_only", arm="global structure",
                       overrides={**base_overrides, "recurrent_connectivity": "within_region"}),
    sweep.ModelVariant(label="cross_plus_self_diagonal", arm="global structure",
                       overrides={**base_overrides,
                                  "recurrent_connectivity": "cross_region_with_self_diagonal"}),
]
for region in REGIONS:
    blocked = tuple(
        [(region, other) for other in REGIONS if other != region]
        + [(other, region) for other in REGIONS if other != region]
    )
    variants.append(sweep.ModelVariant(
        label=f"isolate_{region}", arm="region isolation",
        overrides={**base_overrides, "recurrent_connectivity": "full",
                   "recurrent_blocked_pairs": blocked},
    ))
for source, target in PATHWAYS_TO_TEST:
    variants.append(sweep.ModelVariant(
        label=f"drop_{source}_to_{target}", arm="pathway removal",
        overrides={**base_overrides, "recurrent_connectivity": "full",
                   "recurrent_blocked_pairs": ((source, target),)},
    ))

seeds = protocol.protocol_seeds(n_seeds=SWEEP_SEEDS)
BASELINE = "full"

display(Markdown(f"Width **{HIDDEN_UNITS}** units per region, from {CAPACITY_SOURCE}."))

# What the *trained* runs were fitted at, which may predate task 01's answer. Only runs
# with a checkpoint count: generating the job commands writes a run_config.yaml for every
# cell whether or not it is ever submitted, so staged configs are not evidence of anything.
_trained = sorted(
    path for path in TASK_ROOT.glob("*/seed=*/run_config.yaml")
    if (path.parent / "checkpoint_best.pth").exists()
)
if _trained:
    _widths = {int(yaml.safe_load(path.read_text())["hidden_units"]) for path in _trained}
    if _widths != {HIDDEN_UNITS}:
        display(Markdown(
            f"🔴 **Width mismatch.** {len(_trained)} trained run(s) used {sorted(_widths)} units, "
            f"but the current selection is {HIDDEN_UNITS}. Those runs answer the question at the "
            f"wrong width and should be refitted before the results below are used."
        ))
    else:
        display(Markdown(f"{len(_trained)} trained run(s), all at {HIDDEN_UNITS} units."))
display(pd.DataFrame([{"label": v.label, "arm": v.arm,
                       "connectivity": v.overrides.get("recurrent_connectivity"),
                       "blocked pairs": len(v.overrides.get("recurrent_blocked_pairs", ()))}
                      for v in variants]))
display(Markdown(
    f"**{len(variants)} variants × {len(seeds)} seeds = {len(variants) * len(seeds)} runs** at "
    f"{SELECTED_PROTOCOL['epochs']:,} iterations."
))

Width **40** units per region, from task 01 (`h40`).

,label,arm,connectivity,blocked pairs
0,full,baseline,full,0
1,within_region_only,global structure,within_region,0
2,cross_plus_self_diagonal,global structure,cross_region_with_self_diagonal,0
3,isolate_ofc,region isolation,full,6
4,isolate_bla,region isolation,full,6
5,isolate_dmpfc,region isolation,full,6
6,isolate_accg,region isolation,full,6
7,drop_ofc_to_bla,pathway removal,full,1
8,drop_dmpfc_to_bla,pathway removal,full,1
9,drop_accg_to_bla,pathway removal,full,1


**13 variants × 3 seeds = 39 runs** at 100,000 iterations.

## 2. Run state and submission

Submission happens **only** if you set `SUBMIT = True`, and is blocked while a previously
submitted array is still on the queue.

In [3]:
SUBMIT = False   # <-- set to True to actually submit the missing cells

commands, run_dirs = sweep.variant_job_commands(
    variants, seeds, root=TASK_ROOT, repo_root=repo_root,
    protocol=SELECTED_PROTOCOL, mrnn_cfg_path=MRNN_CFG_PATH,
)
inventory = sweep.index_variant_runs(TASK_ROOT, variants, seeds)
job_state = protocol.running_job_state(TASK_ROOT / "_jobs")

if SELECTED_CAPACITY is None:
    display(Markdown(
        f"⚠️ **Task 01 has not finished**, so these cells would be fitted at the provisional "
        f"width of **{HIDDEN_UNITS}** units. "
        + ("`ALLOW_PROVISIONAL_WIDTH` is set, so submission is allowed — Section 1 will flag a "
           "mismatch once task 01 reports."
           if ALLOW_PROVISIONAL_WIDTH else
           "Submission is blocked; set `ALLOW_PROVISIONAL_WIDTH = True` to queue anyway.")
    ))
display(Markdown(
    f"**{int(inventory['complete'].sum())} complete**, **{int(inventory['diverged'].sum())} diverged**, "
    f"**{int(inventory['pending'].sum())} not yet run** of {len(inventory)} cells."
))
if job_state["active"]:
    display(Markdown(
        f"⚠️ **Job array `{job_state['job_id']}` is still on the queue** "
        f"({', '.join(f'{n} {s.lower()}' for s, n in sorted(job_state['states'].items()))})."
    ))
elif commands:
    display(Markdown(f"{len(commands)} run(s) would be submitted."))
    print("first command:\n")
    print(commands[0])

**0 complete**, **0 diverged**, **39 not yet run** of 39 cells.

39 run(s) would be submitted.

first command:

cd /gpfs/milgram/pi/chang/pg496/repositories/dal_monte_2022_analysis && FIXATION_MRNN_PROGRESS=off conda run -n gaze_processing python scripts/ephys/modeling/train_fixation_mrnn_into_run_dir.py --mrnn-cfg /gpfs/milgram/pi/chang/pg496/repositories/local_data/dal_monte_2022/analysis_outputs/ephys/modeling/fixation_mrnn/chapter/02_connectivity/full/seed=271795885/run_config.yaml --run-dir /gpfs/milgram/pi/chang/pg496/repositories/local_data/dal_monte_2022/analysis_outputs/ephys/modeling/fixation_mrnn/chapter/02_connectivity/full/seed=271795885 --seed 271795885 --device auto --overwrite


In [4]:
if job_state["active"]:
    display(Markdown(f"Nothing submitted: job array `{job_state['job_id']}` is still running."))
elif SUBMIT and commands and SELECTED_CAPACITY is None and not ALLOW_PROVISIONAL_WIDTH:
    display(Markdown(
        f"**Not submitted**: task 01 has not selected a width, so these cells would be fitted at "
        f"the provisional {HIDDEN_UNITS} units. Set `ALLOW_PROVISIONAL_WIDTH = True` to queue them "
        f"anyway — worth doing when the cluster is the bottleneck, since a mismatch is detectable "
        f"afterwards and only costs a refit."
    ))
elif SUBMIT and commands:
    from dal_monte_2022_analysis.runtime.hpc.jobs import submit_dsq_array_job, write_job_file

    jobs_dir = TASK_ROOT / "_jobs"
    jobs_dir.mkdir(parents=True, exist_ok=True)
    job_file = jobs_dir / "connectivity.txt"
    write_job_file(job_file, commands)
    job_id = submit_dsq_array_job(
        job_file_path=job_file, sbatch_script_path=jobs_dir / "connectivity.sh",
        log_dir=jobs_dir / "logs", job_name="mrnn_connectivity", partition="psych_gpu",
        cpus_per_task=1, mem_per_cpu="12G", time_limit="06:00:00", gres="gpu:1",
    )
    (jobs_dir / "job_id.txt").write_text(str(job_id) + "\n")
    display(Markdown(f"Submitted **{len(commands)}** runs as job array **{job_id}**."))
elif commands:
    display(Markdown("`SUBMIT` is **False** — nothing was submitted."))
else:
    display(Markdown("Every cell is already trained; go on to Section 3."))

`SUBMIT` is **False** — nothing was submitted.

## 3. Loss trajectories and convergence

A constraint that merely makes optimisation harder looks the same in the final loss as
one the data cannot tolerate. Only the trajectory separates them, so this comes first.

In [5]:
histories = sweep.load_histories(inventory)
labels = [v.label for v in variants if v.label in histories]

if not histories:
    display(Markdown("No completed runs yet — this section fills in as the sweep lands."))
else:
    convergence = sweep.convergence_table(histories)
    display(convergence.round(5))
    show(viz.plot_sweep_loss_trajectories(histories, convergence=convergence, n_columns=4),
         "fig01_loss_trajectories")

No completed runs yet — this section fills in as the sweep lands.

## 4. Fit, and what it cost

Scored against the measured noise ceiling. The parameter count matters more here than in
task 01: every constraint in this sweep also *removes* parameters, so a variant that fits
as well as the baseline with fewer of them is the interesting outcome, and one that fits
worse has to be checked against how much smaller it is before that is called a structural
result.

In [6]:
pc_ceiling = pd.read_csv(CEILING_DIR / "pc_space_ceiling.csv")
ceiling_by_region = pc_ceiling.groupby("region")["reliability"].mean().to_dict()

if not histories:
    display(Markdown("Nothing to score yet."))
else:
    fit = sweep.score_variant_fit(inventory, ceiling_by_region)
    parameters = pd.DataFrame([
        {"label": label, "arm": inventory[inventory["label"] == label]["arm"].iloc[0],
         **sweep.count_trainable_parameters(
             inventory[(inventory["label"] == label) & inventory["complete"]].iloc[0]["run_dir"])}
        for label in labels
    ])
    summary = (
        fit.groupby("label")["r2_vs_ceiling"].agg(["mean", "min"])
        .join(parameters.set_index("label")[["arm", "inter_region", "total", "parameters_per_datum"]])
        .loc[labels]
    )
    reference = float(summary.loc[BASELINE, "mean"]) if BASELINE in summary.index else np.nan
    summary["cost_vs_baseline"] = reference - summary["mean"]
    display(summary.sort_values("cost_vs_baseline").round(4))
    show(viz.plot_fit_versus_constraint(fit, parameters, order=labels,
                                        x_label="connectivity variant"), "fig02_fit_vs_connectivity")

Nothing to score yet.

In [7]:
if histories:
    by_condition = (
        fit.groupby(["label", "condition"])["r2_vs_ceiling"].mean().unstack().loc[labels]
    )
    display(Markdown(
        "Broken down by condition. The legacy single-seed comparison found the cost of removing "
        "inter-regional coupling several times larger for interactive-face fixations than for the "
        "others; whether that survives refitting across seeds is the question this table answers."
    ))
    display(by_condition.round(4))

### 4a. Region isolation, decomposed

Each isolation is two experiments in one, and the pooled score above mixes them. Split
apart:

- **isolated_cost** — how much worse the cut-off region's *own* trajectories are once it
  has only its condition input and its internal recurrence. A large cost means that region
  depends on the others.
- **remaining_cost** — how much worse the *other three* are without it. A large cost means
  that region is necessary to them.

The second column is the one that speaks to network structure. A region can score badly on
the first and not at all on the second — that would say it is driven by the network rather
than driving it.

In [8]:
if histories:
    isolated_by_label = {
        v.label: v.label.replace("isolate_", "")
        for v in variants if v.arm == "region isolation" and v.label in labels
    }
    if isolated_by_label and BASELINE in labels:
        isolation = sweep.decompose_isolation_fit(
            fit, isolated_region_by_label=isolated_by_label, baseline_label=BASELINE
        )
        display(isolation.round(4))
        worst_self = isolation.loc[isolation["isolated_cost"].idxmax()]
        worst_others = isolation.loc[isolation["remaining_cost"].idxmax()]
        display(Markdown(
            f"The region that suffers most from being cut off is **{worst_self['isolated_region']}** "
            f"(its own fit falls {float(worst_self['isolated_cost']):.4f}), and the region the others "
            f"miss most is **{worst_others['isolated_region']}** (their fit falls "
            f"{float(worst_others['remaining_cost']):.4f}). Where those are different regions, the "
            f"network is asymmetric: one region is a listener and another a driver."
        ))
    else:
        display(Markdown("Needs the baseline and at least one isolation variant."))

## 5. The visual check

One row per variant, all showing the same three components of the same region. A
constraint can leave $R^2$ almost untouched and still visibly change the shape of the
trajectory, which is exactly what happened to interactive face in task 00.

In [9]:
if histories:
    pc_traces = sweep.gallery_traces(inventory, region=GALLERY_REGION, space="pc", indices=(0, 1, 2))
    show(viz.plot_fit_gallery(pc_traces, order=labels,
                              title=f"{GALLERY_REGION.upper()} — top three PCs, observed against mRNN"),
         "fig03_gallery_pc")

## 6. Seed agreement

The axis on which a constrained model can beat the unconstrained one. Removing
connections removes ways of producing the same output, so if any structural constraint
makes the solution more identifiable, it should show here — and identifiability is what
every downstream circuit claim needs.

In [10]:
if histories:
    agreement = pd.concat([
        tl.inter_seed_agreement(list(block["run_dir"])).assign(label=label)
        for label, block in inventory[inventory["complete"].astype(bool)].groupby("label", sort=False)
    ], ignore_index=True)
    display(agreement.pivot_table(index="label", columns="feature", values="mean_agreement")
            .loc[labels].round(4))
    show(viz.plot_seed_agreement_by_variant(agreement, order=labels), "fig04_seed_agreement")

## 7. Reading the result

Nothing is "selected" here — connectivity is the object of study, not a hyperparameter.
What the sweep produces is a statement about which constraints the data tolerates, and it
is written to `connectivity_findings.csv` for the chapter.

The three readings to make, in order:

1. **Which removals are free?** A variant within the seed-to-seed spread of the baseline
   removes connections the data does not need. Those are the strongest claims available
   here, because they survive refitting. For the isolation arm, read the decomposed costs
   in Section 4a rather than the pooled score — a region whose removal looks expensive may
   simply be hard to reproduce alone.
2. **Which removals cost, and does the cost concentrate in one condition?** A structural
   requirement that appears only during interactive-face fixations would be the
   substantive result.
3. **Does any constraint raise seed agreement?** If a smaller model is more identifiable
   at no cost in fit, that is the model tasks 05–07 should be built on.

In [11]:
if not histories:
    display(Markdown("Deferred until the sweep completes."))
else:
    spread = float(fit[fit["label"] == BASELINE].groupby("seed")["r2_vs_ceiling"].mean().std())
    geometry = agreement[agreement["feature"] == "latent drive geometry"].set_index("label")["mean_agreement"]
    findings = summary.join(geometry.rename("seed_agreement_geometry"))
    findings["free_removal"] = findings["cost_vs_baseline"] <= 2 * spread
    findings["more_identifiable"] = (
        findings["seed_agreement_geometry"] > float(geometry.get(BASELINE, np.nan))
    )
    findings.to_csv(TASK_ROOT / "connectivity_findings.csv")
    display(findings.round(4))
    free = [str(i) for i in findings.index[findings["free_removal"]] if str(i) != BASELINE]
    display(Markdown(
        f"Baseline seed-to-seed spread in ceiling-relative fit is **{spread:.4f}**, so a removal is "
        f"called free when it costs less than twice that.\n\n"
        f"**Free removals:** {', '.join(free) if free else 'none'}.\n\n"
        f"Written to `{TASK_ROOT / 'connectivity_findings.csv'}`."
    ))

Deferred until the sweep completes.

## 8. What comes next

**Next:** `03_bottleneck_rank.ipynb`. This task establishes *which* inter-regional blocks
have to exist; task 03 asks how much has to pass through the ones that do, by constraining
each surviving block to rank $r$ and measuring against the dense baseline fitted here.

The two together are the structural claim the chapter rests on, and both are stated as
"the data still supports this under constraint X" rather than as a model ranking.